# A. Initialization and depth, in PyTorch

The PyTorch counterpart of GlassNN example `02_init_and_depth` (lecture 10,
book chapter 4). Run it on [Google Colab](https://colab.research.google.com),
where PyTorch is preinstalled.

**Goals**

- Write the same deep MLP and the same five initializations with `torch.nn`
  and `torch.nn.init`: the code is almost line by line the GlassNN code.
- Replace `glassnn.analysis.activation_stats` by PyTorch *forward hooks*.

**Expected finding** (the same as in GlassNN): in a 20-layer ReLU network of
width 256, only Kaiming initialization keeps the scale of the
pre-activations constant; the others make it vanish or explode
geometrically. The numbers differ slightly from the GlassNN notebook because
the two libraries draw different random weights.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import init

WIDTH, DEPTH = 256, 20
X = torch.from_numpy(np.random.default_rng(0).normal(size=(512, WIDTH))).float()

INITIALIZERS = {
    "too small, N(0, 0.01²)": lambda w: init.normal_(w, std=0.01),
    "default of Linear": lambda w: init.kaiming_uniform_(w, a=math.sqrt(5)),
    "Xavier normal": init.xavier_normal_,
    "Kaiming normal (relu)": lambda w: init.kaiming_normal_(w, nonlinearity="relu"),
    "too large, N(0, 0.1²)": lambda w: init.normal_(w, std=0.1),
}


def deep_mlp(activation, initializer):
    # DEPTH linear layers of width WIDTH, each followed by the activation.
    torch.manual_seed(0)
    layers = []
    for _ in range(DEPTH):
        layer = nn.Linear(WIDTH, WIDTH)
        initializer(layer.weight)
        init.zeros_(layer.bias)
        layers += [layer, activation()]
    return nn.Sequential(*layers)

PyTorch has no `activation_stats`. A *forward hook* is a function that
PyTorch calls after a module's `forward`, with the module's input and output;
here it records the standard deviation of the output $Z_l$ of every `Linear`
layer. `torch.no_grad()` plays the role of `glassnn.no_grad()`.

In [ ]:
def output_stds(model, X):
    stds = []

    def record(module, inputs, output):
        stds.append(output.std().item())

    hooks = [m.register_forward_hook(record) for m in model if isinstance(m, nn.Linear)]
    with torch.no_grad():
        model(X)
    for hook in hooks:
        hook.remove()
    return stds


output_stds(deep_mlp(nn.ReLU, init.kaiming_normal_), X)[:3]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
ratios = {}
for ax, activation in zip(axes, [nn.ReLU, nn.Tanh], strict=True):
    for label, initializer in INITIALIZERS.items():
        stds = output_stds(deep_mlp(activation, initializer), X)
        ratios[(activation.__name__, label)] = stds[-1] / stds[0]
        ax.semilogy(range(1, DEPTH + 1), stds, marker=".", label=label)
    ax.set_title(activation.__name__)
    ax.set_xlabel("layer $l$")
axes[0].set_ylabel("std of $Z_l$")
axes[1].legend(fontsize=8)
plt.show()

for (activation, label), ratio in ratios.items():
    print(f"{activation:5s} {label:25s} std(Z_20) / std(Z_1) = {ratio:.2e}")

**Finding.** As in GlassNN: with ReLU only Kaiming keeps the ratio close
to 1; Xavier loses a factor $1/\sqrt 2$ per layer and PyTorch's default a
factor $1/\sqrt 6$. GlassNN's `nn.Linear` has the same default
initialization as `torch.nn.Linear`, which is why the curves match.

## Rosetta table: GlassNN and PyTorch

Names, arguments and behaviour that differ between the two libraries (the same table as in chapter 12 of the GlassNN book). Everything not listed has the same name and meaning.

| Area | GlassNN | PyTorch | Difference |
|---|---|---|---|
| Tensors | `Tensor` | `torch.Tensor` | A float64 NumPy array is cast to the default dtype (PyTorch keeps float64). The graph is kept after `backward()`, so a second call works without `retain_graph=True`. There are no in-place operations (`+=`, item assignment); they raise an error. |
| Tensors | `Tensor.numpy` | `torch.Tensor.numpy()` | Returns a copy; PyTorch shares memory with the tensor. |
| Tensors | `backend.asarray` | `torch.as_tensor` | Floating-point input becomes the default dtype, so float64 data never mixes silently with float32 parameters. |
| Device and randomness | `backend.set_backend` | `tensor.to("cuda")`, `model.to("cuda")` | One global switch: all tensors created afterwards are on the GPU (CuPy). PyTorch places each tensor on a device. |
| Device and randomness | `backend.manual_seed` | `torch.manual_seed` | Returns a `numpy.random.Generator`, the same on CPU and GPU, so a seed gives identical runs on both; it is unseeded until called. PyTorch returns a `torch.Generator`, starts from a fixed seed, and draws different numbers on each device. |
| Device and randomness | `backend.from_dlpack`, `backend.to_dlpack` | `torch.from_dlpack`, `torch.utils.dlpack.to_dlpack` | Work on arrays, not tensors: wrap the result with `Tensor(...)`, pass `tensor.data`. |
| Data | `data.DataLoader` | `torch.utils.data.DataLoader` | Takes the arrays `X, y` directly instead of a `Dataset`; no workers, samplers or `collate_fn`. |
| Data | `data.one_hot` | `torch.nn.functional.one_hot` | Lives in `glassnn.data`. |
| Modules | `nn.Module.state_dict` | `Module.state_dict()` | Returns copies of the values, not tensors that share memory; parameters and buffers are listed module by module. |
| Modules | `nn.Module.register_buffer` | `Module.register_buffer(..., persistent=True)` | No `persistent` argument: every buffer is saved. |
| Layers | `nn.Linear` | `nn.Linear` | Extra argument `parametrization="ntk"` (NTK scaling, chapter 8); the default initialization is PyTorch's. |
| Layers | `nn.Conv1d`, `nn.Conv2d` | `nn.Conv1d`, `nn.Conv2d` | No `groups` and no `padding_mode` (zero padding only). |
| Layers | `functional.conv1d`, `functional.conv2d` | `F.conv1d`, `F.conv2d` | No `groups`; zero padding only. |
| Layers | `functional.max_pool1d`, `functional.max_pool2d`, `nn.MaxPool1d`, `nn.MaxPool2d` | `F.max_pool1d`, `nn.MaxPool1d`, ... | No `dilation`, `ceil_mode` or `return_indices`. |
| Layers | `functional.avg_pool1d`, `functional.avg_pool2d`, `nn.AvgPool1d`, `nn.AvgPool2d` | `F.avg_pool1d`, `nn.AvgPool1d`, ... | No `ceil_mode`; padding always counts in the mean (`count_include_pad=True`). |
| Layers | `nn.BatchNorm1d` | `nn.BatchNorm1d` | No `num_batches_tracked` buffer and no `momentum=None` (cumulative average). |
| Layers | `nn.Dropout` | `nn.Dropout` | No `inplace` argument. |
| Layers | `functional.dropout` | `F.dropout` | No `inplace` argument; an extra `generator=` argument. |
| Attention | `functional.scaled_dot_product_attention` | `F.scaled_dot_product_attention` | A boolean mask is `True` where attention is **forbidden** (PyTorch, in this function only: `True` = allowed). |
| Attention | `nn.MultiheadAttention` | `nn.MultiheadAttention` | Batch-first only, `(N, L, E)`. Three projections `q_proj`, `k_proj`, `v_proj` instead of one packed `in_proj_weight`, so `state_dict` names differ. No `kdim`, `vdim`, `add_bias_kv`, `add_zero_attn`. `is_causal=True` without a mask builds the causal mask. |
| Attention | `nn.TransformerEncoderLayer` | `nn.TransformerEncoderLayer` | Batch-first only. Defaults `norm_first=True` and `activation="gelu"` (PyTorch: `False`, `"relu"`); `activation` is a string. |
| Attention | `nn.TransformerEncoder` | `nn.TransformerEncoder` | No `enable_nested_tensor` or `mask_check`. |
| Attention | `nn.PositionalEncoding` | none (defined in PyTorch tutorials) | A GlassNN module: sinusoidal, batch-first, dropout 0 by default. |
| Losses | `functional.cross_entropy`, `nn.CrossEntropyLoss` | `F.cross_entropy`, `nn.CrossEntropyLoss` | Only `(N, C)` logits with integer targets: no class probabilities, `weight` or `ignore_index`. |
| Losses | `functional.mse_loss` | `F.mse_loss` | Different shapes raise an error (PyTorch broadcasts with a warning); `binary_cross_entropy_with_logits` does the same. |
| Optimizers | `optim.Optimizer` | `torch.optim.Optimizer` | One parameter group; the rate is `optimizer.lr` (PyTorch: `optimizer.param_groups[0]["lr"]`). |
| Optimizers | `optim.LinearWarmup` | `lr_scheduler.LinearLR(optimizer, start_factor, end_factor=1.0, total_iters=warmup_steps)` | The same schedule under a different name; all schedulers are also in `glassnn.optim`. |
| Optimizers | `optim.clip_grad_norm_` | `torch.nn.utils.clip_grad_norm_` | Lives in `glassnn.optim`; Euclidean norm only. |
| Tools | `gradcheck.gradcheck` | `torch.autograd.gradcheck` | Checks one vector-Jacobian product with fixed weights instead of the full Jacobian, and raises an error that names the element and both values. |
| Tools | `analysis.empirical_ntk`, `analysis.activation_stats`, `analysis.count_parameters` | none (`torch.func.jacrev`, forward hooks, `sum(p.numel() ...)`) | GlassNN helpers for chapters 4 and 8; the Colab notebooks show the PyTorch equivalents. |